
# Protein Secondary Structure Prediction with CNN

## Goal
This notebook trains a **Convolutional Neural Network (CNN)** to predict protein secondary structure (3‑state: H, E, C) using two different sequence encodings:
1. **One‑hot encoding** – classical residue‑level representation.
2. **ESM‑2 embeddings** – deep contextualised features from a pretrained protein language model.
 
The CNN operates on sliding windows of fixed size. For each window, the input is a 2D matrix of shape `(window_size, embedding_dim)`. The CNN applies 1D convolutions along the window length to extract local spatial features, followed by pooling and fully connected layers.

## Dataset
Same as before: [Protein Secondary Structure (CASP12, CB513, TS115)](https://www.kaggle.com/datasets/tamzidhasan/protein-secondary-structure-casp12-cb513-ts115).  
We use the **`sst3`** column (3‑state: H, E, C). The data are already split into train / validation / test.

## CNN Architecture
- **Input shape**: `(window_size, embedding_dim)`
- **Conv1D layer**: 64 filters, kernel size 3, ReLU activation
- **MaxPooling1D**: pool size 2
- **Conv1D layer**: 128 filters, kernel size 3, ReLU activation
- **GlobalMaxPooling1D** (to reduce to a fixed‑size vector)
- **Dropout(0.5)**
- **Dense layer**: 64 units, ReLU
- **Output layer**: 3 units (softmax)

The model is trained with cross‑entropy loss and Adam optimizer.

In [9]:
# Import standard libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import sys
from pathlib import Path
import os

project_root = Path.cwd().parent.parent
sys.path.append(str(project_root))

# print("Project root:", project_root)  
data_root = project_root / "data" / "raw"

# Our custom embedding functions
from embeddings import prepare_data_onehot, prepare_data_esm_chunked

# ESM library
import esm

# For metrics and plots
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns


In [10]:
# Check for GPU
def get_default_device():
    """Return 'cuda' only if a compatible GPU is available."""
    if torch.cuda.is_available():
        cap = torch.cuda.get_device_capability()
        if cap[0] >= 7:
            return torch.device('cuda')
        else:
            print(f"GPU compute capability {cap[0]}.{cap[1]} is < 7.0. Falling back to CPU.")
    return torch.device('cpu')

device = get_default_device()
print(f"Using device: {device}")

Using device: cpu


In [11]:
train_df = pd.read_csv(data_root / 'training_secondary_structure_train.csv')
valid_df = pd.read_csv(data_root / 'validation_secondary_structure_valid.csv')
test_df = pd.read_csv(data_root / 'test_secondary_structure_casp12.csv')


window_size = 15
print(f"Training proteins: {len(train_df)}")
print(f"Validation proteins: {len(valid_df)}")
print(f"Test (CASP12) proteins: {len(test_df)}")
print("Columns:", train_df.columns.tolist())

# Extract sequences and labels
train_seqs = train_df['seq'].tolist()
train_ss3 = train_df['sst3'].tolist()
valid_seqs = valid_df['seq'].tolist()
valid_ss3 = valid_df['sst3'].tolist()
test_seqs = test_df['seq'].tolist()
test_ss3 = test_df['sst3'].tolist()

Training proteins: 8678
Validation proteins: 2170
Test (CASP12) proteins: 21
Columns: ['seq', 'sst3', 'sst8']


In [12]:
class SimpleCNN(nn.Module):
    def __init__(self, input_channels, window_size, num_classes=3):
        super().__init__()
        self.window_size = window_size
        
        self.conv1 = nn.Conv1d(in_channels=input_channels, out_channels=64, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        
        self.global_pool = nn.AdaptiveAvgPool1d(1)  # outputs (batch, channels, 1)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(128, 64)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        # x shape: (batch, window_size, input_channels) -> (batch, input_channels, window_size)
        x = x.transpose(1, 2)
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        x = self.global_pool(x)          # (batch, 128, 1)
        x = x.squeeze(-1)                # (batch, 128)
        x = self.dropout(x)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)
        return x

In [13]:
def train_and_evaluate_cnn(X_train, y_train, X_valid, y_valid, X_test, y_test,
                           input_channels, window_size, epochs=20, batch_size=64, lr=0.001, device=None):
    if device is None:
        device = get_default_device()
    print(f"Using device: {device}")
    
    model = SimpleCNN(input_channels, window_size).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
    X_valid_t = torch.tensor(X_valid, dtype=torch.float32).to(device)
    y_valid_t = torch.tensor(y_valid, dtype=torch.long).to(device)
    
    train_dataset = TensorDataset(X_train_t, y_train_t)
    valid_dataset = TensorDataset(X_valid_t, y_valid_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
    
    best_val_acc = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_x, batch_y in valid_loader:
                outputs = model(batch_x)
                _, preds = torch.max(outputs, 1)
                correct += (preds == batch_y).sum().item()
                total += batch_y.size(0)
        val_acc = correct / total
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1:2d} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")
    
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_t)
        _, preds = torch.max(outputs, 1)
        test_acc = (preds == y_test_t).float().mean().item()
    print(f"Test Accuracy: {test_acc:.4f}")
    return test_acc, best_val_acc, model


In [14]:
print("=== One-hot encoding (CNN) ===")
X_train_oh_flat, y_train_oh = prepare_data_onehot(train_seqs, train_ss3, window_size)
X_valid_oh_flat, y_valid_oh = prepare_data_onehot(valid_seqs, valid_ss3, window_size)
X_test_oh_flat, y_test_oh   = prepare_data_onehot(test_seqs, test_ss3, window_size)

# Reshape to (n_samples, window_size, 20)
X_train_oh = X_train_oh_flat.reshape(-1, window_size, 20)
X_valid_oh = X_valid_oh_flat.reshape(-1, window_size, 20)
X_test_oh  = X_test_oh_flat.reshape(-1, window_size, 20)

print(f"One-hot CNN input shape: {X_train_oh.shape}")


=== One-hot encoding (CNN) ===
One-hot CNN input shape: (2221507, 15, 20)


In [15]:
# Train CNN
input_channels_oh = 20
acc_oh_cnn, val_oh_cnn, model_oh_cnn = train_and_evaluate_cnn(
    X_train_oh, y_train_oh, X_valid_oh, y_valid_oh, X_test_oh, y_test_oh,
    input_channels_oh, window_size, epochs=15
)

Using device: cpu
Epoch  5 | Loss: 0.7606 | Val Acc: 0.6629
Epoch 10 | Loss: 0.7551 | Val Acc: 0.6692
Epoch 15 | Loss: 0.7527 | Val Acc: 0.6694
Test Accuracy: 0.6621


## 6. Prepare ESM Data (CNN format)


In [ ]:
print("\n=== ESM embeddings (CNN) ===")
subset_size = 200   # change to None to use all
if subset_size:
    train_seqs_sub = train_seqs[:subset_size]
    train_ss3_sub = train_ss3[:subset_size]
    valid_seqs_sub = valid_seqs[:subset_size]
    valid_ss3_sub = valid_ss3[:subset_size]
    test_seqs_sub = test_seqs[:subset_size]
    test_ss3_sub = test_ss3[:subset_size]
else:
    train_seqs_sub = train_seqs
    train_ss3_sub = train_ss3
    valid_seqs_sub = valid_seqs
    valid_ss3_sub = valid_ss3
    test_seqs_sub = test_seqs
    test_ss3_sub = test_ss3

# Load ESM model
esm_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
esm_model = esm_model.to(device)
batch_converter = alphabet.get_batch_converter()

# Extract embeddings (flattened windows)
X_train_esm_flat, y_train_esm = prepare_data_esm_chunked(
    train_seqs_sub, train_ss3_sub, esm_model, batch_converter, device, window_size, chunk_size=50)
X_valid_esm_flat, y_valid_esm = prepare_data_esm_chunked(
    valid_seqs_sub, valid_ss3_sub, esm_model, batch_converter, device, window_size, chunk_size=50)
X_test_esm_flat, y_test_esm = prepare_data_esm_chunked(
    test_seqs_sub, test_ss3_sub, esm_model, batch_converter, device, window_size, chunk_size=50)

# Reshape to (n_samples, window_size, 1280)
X_train_esm = X_train_esm_flat.reshape(-1, window_size, 1280)
X_valid_esm = X_valid_esm_flat.reshape(-1, window_size, 1280)
X_test_esm  = X_test_esm_flat.reshape(-1, window_size, 1280)

print(f"ESM CNN input shape: {X_train_esm.shape}")

In [ ]:
# train esm embeddings with cnn model
input_channels_esm = 1280
acc_esm_cnn, val_esm_cnn, model_esm_cnn = train_and_evaluate_cnn(
    X_train_esm, y_train_esm, X_valid_esm, y_valid_esm, X_test_esm, y_test_esm,
    input_channels_esm, window_size, epochs=15
)


In [ ]:
# ------------------------------------------------------------
# 7. ProtT5 Embeddings (CNN)
# ------------------------------------------------------------
print("\n=== ProtT5 Embeddings (CNN) ===")

# Optional subset for speed (change to None to use all data)
subset_size = 200
if subset_size:
    train_seqs_pt5 = train_seqs[:subset_size]
    train_ss3_pt5 = train_ss3[:subset_size]
    valid_seqs_pt5 = valid_seqs[:subset_size]
    valid_ss3_pt5 = valid_ss3[:subset_size]
    test_seqs_pt5 = test_seqs[:subset_size]
    test_ss3_pt5 = test_ss3[:subset_size]
else:
    train_seqs_pt5 = train_seqs
    train_ss3_pt5 = train_ss3
    valid_seqs_pt5 = valid_seqs
    valid_ss3_pt5 = valid_ss3
    test_seqs_pt5 = test_seqs
    test_ss3_pt5 = test_ss3

# Load ProtT5 model (downloads on first run)
print("Loading ProtT5 model... (this may take a few minutes on first run)")
prott5_model, prott5_tokenizer = get_prott5_model(device)

# Extract embeddings (chunked to manage memory)
X_train_pt5, y_train_pt5 = prepare_data_prott5_chunked(
    train_seqs_pt5, train_ss3_pt5, prott5_model, prott5_tokenizer, 
    device, window_size, chunk_size=50)
X_valid_pt5, y_valid_pt5 = prepare_data_prott5_chunked(
    valid_seqs_pt5, valid_ss3_pt5, prott5_model, prott5_tokenizer, 
    device, window_size, chunk_size=50)
X_test_pt5, y_test_pt5 = prepare_data_prott5_chunked(
    test_seqs_pt5, test_ss3_pt5, prott5_model, prott5_tokenizer, 
    device, window_size, chunk_size=50)

# X shape is already (n_windows, window_size, 1024) – no need to reshape
input_channels_pt5 = 1024
acc_pt5, val_pt5, model_pt5 = train_cnn(
    X_train_pt5, y_train_pt5, X_valid_pt5, y_valid_pt5,
    X_test_pt5, y_test_pt5, input_channels_pt5, window_size, epochs=15
)

# Evaluate ProtT5 model
cm_pt5, preds_pt5 = evaluate_model(model_pt5, X_test_pt5, y_test_pt5, device, "ProtT5 CNN")

In [ ]:
# ------------------------------------------------------------
# 8. Final Comparison (One-hot, ESM-2, ProtT5)
# ------------------------------------------------------------
print("\n=== Final CNN Results ===")
print(f"One-hot CNN: Best Val = {val_oh:.4f}, Test Acc = {acc_oh:.4f}")
print(f"ESM-2 CNN:   Best Val = {val_esm:.4f}, Test Acc = {acc_esm:.4f}")
print(f"ProtT5 CNN:  Best Val = {val_pt5:.4f}, Test Acc = {acc_pt5:.4f}")

# Update per‑class accuracy plot to include ProtT5
classes = ['H', 'E', 'C']
per_class_oh = [accuracy_score(y_test_oh[y_test_oh==i], preds_oh[y_test_oh==i]) for i in range(3)]
per_class_esm = [accuracy_score(y_test_esm[y_test_esm==i], preds_esm[y_test_esm==i]) for i in range(3)]
per_class_pt5 = [accuracy_score(y_test_pt5[y_test_pt5==i], preds_pt5[y_test_pt5==i]) for i in range(3)]

# Plot three bars per class
x = np.arange(3)
width = 0.25
plt.figure(figsize=(10,6))
plt.bar(x - width, per_class_oh, width, label='One-hot', color='lightgreen')
plt.bar(x, per_class_esm, width, label='ESM-2', color='coral')
plt.bar(x + width, per_class_pt5, width, label='ProtT5', color='skyblue')
plt.ylabel('Accuracy')
plt.xlabel('Class')
plt.title('Per‑class Accuracy Comparison (CNN)')
plt.xticks(x, classes)
plt.legend()
plt.ylim(0,1)

# Add value labels
for i, (oh, esm, pt5) in enumerate(zip(per_class_oh, per_class_esm, per_class_pt5)):
    plt.text(i - width, oh + 0.02, f'{oh:.3f}', ha='center', fontsize=9)
    plt.text(i, esm + 0.02, f'{esm:.3f}', ha='center', fontsize=9)
    plt.text(i + width, pt5 + 0.02, f'{pt5:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# Confusion matrices for all three models
fig, axes = plt.subplots(1, 3, figsize=(15,5))
sns.heatmap(cm_oh, annot=True, fmt='d', cmap='Greens', xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title('One-hot CNN')
sns.heatmap(cm_esm, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title('ESM-2 CNN')
sns.heatmap(cm_pt5, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=axes[2])
axes[2].set_title('ProtT5 CNN')
plt.tight_layout()
plt.show()